# Prompt Template Sensitivity Study
## VLM Medical VQA Benchmark

**Purpose:** Evaluate whether the v2 prompt template (derived from the MedGemma Technical Report)
is optimal for HuatuoGPT-7B and LLaVA-Med-7B, or whether these architectures respond
better to simpler instruction formats.

**Design:**
- Dataset: SLAKE EN test split, 200-sample stratified subset (seed=42, same as few-shot experiment)
- Models: HuatuoGPT-Vision-7B (Qwen2.5VL backbone) and LLaVA-Med-1.5-Mistral-7B
- Prompt variants: v2_baseline, v3_simple, v4_direct
- Run one model per Kaggle session (GPU RAM constraint)
- Output: one JSONL file per model × variant → 3 files per run (6 total)

**Variants tested:**

| Variant | Description |
|---|---|
| v2_baseline | MedGemma paper prompt with `Final Answer: X` extraction anchor |
| v3_simple | Minimal instruction: question + "Answer concisely." No anchoring. |
| v4_direct | Instruction-following format: "Answer:" prefix instead of "Final Answer:" |


In [ ]:
# Cell 1 — Install dependencies
!pip install -q transformers==4.51.3 datasets accelerate bitsandbytes sentencepiece pillow
import os, json, re
from tqdm.auto import tqdm
os.makedirs('/kaggle/working/outputs', exist_ok=True)


In [ ]:
# Cell 2 — Configuration
# ── CHANGE THIS before each run ──────────────────────────────────────
MODEL_ID = 'FreedomIntelligence/HuatuoGPT-Vision-7B-Qwen2.5VL'
# MODEL_ID = 'microsoft/llava-med-v1.5-mistral-7b'
# ─────────────────────────────────────────────────────────────────────

OUTPUT_DIR    = '/kaggle/working/outputs'
SUBSET_SIZE   = 200
SEED          = 42
DATASET_NAME  = 'BoKelvin/SLAKE'
MODEL_SAFE    = MODEL_ID.replace('/', '_')

print(f"Model   : {MODEL_ID}")
print(f"Output  : {OUTPUT_DIR}")


In [ ]:
# Cell 3 — Build 200-sample stratified subset (identical to few-shot experiment)
import random
from datasets import load_dataset

ds = load_dataset(DATASET_NAME, split='test')
en_records = [s for s in ds if s.get('q_lang') == 'en']

# Stratify by content_type × answer_type (6 buckets, ~33 each)
from collections import defaultdict
buckets = defaultdict(list)
for i, s in enumerate(en_records):
    key = (s['content_type'], s['answer_type'])
    buckets[key].append((i, s))

random.seed(SEED)
subset = []
per_bucket = SUBSET_SIZE // len(buckets)
for key, items in buckets.items():
    random.shuffle(items)
    subset.extend(items[:per_bucket])

# Top up to exactly 200 if needed
all_remaining = [(i, s) for key, items in buckets.items() for i, s in items[per_bucket:]]
random.shuffle(all_remaining)
subset.extend(all_remaining[:SUBSET_SIZE - len(subset)])
subset = sorted(subset, key=lambda x: x[0])  # restore original order

print(f"Subset size: {len(subset)}")
from collections import Counter
print("Content types:", Counter(s['content_type'] for _, s in subset))
print("Answer types:", Counter(s['answer_type'] for _, s in subset))


In [ ]:
# Cell 4 — Prompt variants
# v2_baseline: MedGemma paper prompt (Final Answer: X anchor)
# v3_simple:   Question only + minimal instruction (no anchor)
# v4_direct:   Answer: prefix instead of Final Answer:

def build_v2_baseline(question, is_closed):
    prefix = 'Answer the question with yes or no. ' if is_closed else ''
    return (
        f"{prefix}{question} "
        f"You may write out your argument before stating your final very short, "
        f"definitive, and concise answer (if possible, a single word) "
        f"X in the format 'Final Answer: X'"
    )

def build_v3_simple(question, is_closed):
    prefix = 'Answer with yes or no only. ' if is_closed else ''
    return f"{prefix}{question} Answer concisely in one word or short phrase."

def build_v4_direct(question, is_closed):
    prefix = 'Answer with yes or no only. ' if is_closed else ''
    return (
        f"{prefix}Look at the medical image and answer the following question in one "
        f"concise word or phrase. Start your response with 'Answer:'.\n"
        f"Question: {question}"
    )

def extract_answer(raw, variant):
    if variant == 'v2_baseline':
        m = re.search(r'[Ff]inal\s*[Aa]nswer\s*:\s*(.+)', raw, re.DOTALL)
        if m:
            ans = m.group(1).strip().split('\n')[0]
            return re.sub(r'[\*\"\']+'  , '', ans).strip()
    elif variant == 'v4_direct':
        m = re.search(r'[Aa]nswer\s*:\s*(.+)', raw, re.DOTALL)
        if m:
            ans = m.group(1).strip().split('\n')[0]
            return re.sub(r'[\*\"\']+'  , '', ans).strip()
    # Fallback: first non-empty line
    lines = [l.strip() for l in raw.split('\n') if l.strip()]
    return lines[0] if lines else raw.strip()

VARIANTS = {
    'v2_baseline': build_v2_baseline,
    'v3_simple':   build_v3_simple,
    'v4_direct':   build_v4_direct,
}

print("Prompt variants defined:", list(VARIANTS.keys()))


In [ ]:
# Cell 5 — Load model
# HuatuoGPT uses a Qwen2.5-VL backbone → needs AutoModelForImageTextToText
# LLaVA-Med uses a Mistral backbone    → needs AutoModelForCausalLM
import torch
from transformers import (
    AutoProcessor,
    AutoModelForCausalLM,
    AutoModelForImageTextToText,
    BitsAndBytesConfig,
)

IS_QWEN_BACKBONE = 'HuatuoGPT' in MODEL_ID or 'Qwen2.5VL' in MODEL_ID or 'Qwen2_5' in MODEL_ID

print(f"Loading {MODEL_ID} ...")
print(f"Backbone type: {'Qwen2.5-VL → AutoModelForImageTextToText' if IS_QWEN_BACKBONE else 'Mistral/LLaVA → AutoModelForCausalLM'}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

if IS_QWEN_BACKBONE:
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        device_map='auto',
        quantization_config=bnb_config,
        trust_remote_code=True,
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map='auto',
        quantization_config=bnb_config,
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )

model.eval()
param_b = sum(p.numel() for p in model.parameters()) / 1e9
print(f"Model loaded. Parameters: {param_b:.2f}B")


In [ ]:
# Cell 6 — Inference loop (all 3 variants)
import re, json, os

for variant_name, builder_fn in VARIANTS.items():
    out_path = f'{OUTPUT_DIR}/{MODEL_SAFE}__{variant_name}.jsonl'
    print(f"\n=== Variant: {variant_name} ===")

    # Resume support: load already-completed records
    completed = {}
    if os.path.exists(out_path):
        with open(out_path) as f_in:
            for line in f_in:
                try:
                    r = json.loads(line)
                    completed[r['idx']] = r
                except: pass
        print(f"  Resuming: {len(completed)} / {len(subset)} already done.")

    with open(out_path, 'a') as f_out:
        for orig_idx, sample in tqdm(subset, desc=variant_name):
            if orig_idx in completed:
                continue

            is_closed = sample['answer_type'] == 'CLOSED'
            question  = sample['question']
            gt        = str(sample['answer'])
            prompt_text = builder_fn(question, is_closed)

            messages = [{
                'role': 'user',
                'content': [
                    {'type': 'image', 'image': sample['image']},
                    {'type': 'text',  'text': prompt_text},
                ]
            }]

            try:
                text   = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                inputs = processor(text=text, images=sample['image'], return_tensors='pt').to(model.device)
                with torch.inference_mode():
                    out_ids = model.generate(**inputs, max_new_tokens=50, do_sample=False)
                in_len  = inputs['input_ids'].shape[-1]
                raw_out = processor.decode(out_ids[0][in_len:], skip_special_tokens=True).strip()
                pred    = extract_answer(raw_out, variant_name)
            except Exception as e:
                print(f"  Error idx={orig_idx}: {e}")
                raw_out, pred = str(e)[:80], ''

            f_out.write(json.dumps({
                'idx':          orig_idx,
                'question':     question,
                'ground_truth': gt,
                'prediction':   pred,
                'raw_output':   raw_out,
                'is_closed':    is_closed,
                'content_type': sample['content_type'],
                'model':        MODEL_ID,
                'variant':      variant_name,
            }) + '\n')

    print(f"  Saved: {out_path}")


In [ ]:
# Cell 7 — Quick scoring (Token F1 per variant)
import re, json
from collections import Counter

def norm(t):
    t = re.sub(r'[^\w\s]', ' ', str(t).lower())
    return re.sub(r'\s+', ' ', t).strip()

def token_f1(pred, gt):
    p, g = norm(pred).split(), norm(gt).split()
    if not p or not g: return 0.0
    pc, gc = Counter(p), Counter(g)
    common = sum((pc & gc).values())
    if common == 0: return 0.0
    return 2 * common / (len(p) + len(g))

print(f"\nModel: {MODEL_ID}\n")
print(f"{'Variant':<16} {'F1':>7} {'ClsAcc':>8} {'OpenF1':>8} {'N':>5}")
print("-" * 45)

for variant_name in VARIANTS:
    path = f'{OUTPUT_DIR}/{MODEL_SAFE}__{variant_name}.jsonl'
    if not os.path.exists(path): continue
    records = [json.loads(l) for l in open(path)]
    f1s, cls, opn = [], [], []
    for r in records:
        f1 = token_f1(r['prediction'], r['ground_truth'])
        f1s.append(f1)
        if r['is_closed']:
            yn_pred = norm(r['prediction']).split()[0] if norm(r['prediction']).split() else ''
            yn_gt   = norm(r['ground_truth']).split()[0] if norm(r['ground_truth']).split() else ''
            cls.append(1 if yn_pred == yn_gt else 0)
        else:
            opn.append(f1)
    avg = lambda l: sum(l)/len(l)*100 if l else 0
    print(f"{variant_name:<16} {avg(f1s):>6.2f}% {avg(cls):>7.2f}% {avg(opn):>7.2f}% {len(records):>5}")


## After running on Kaggle

Download the 3 output JSONL files (one per variant) and place them in:
```
outputs/_archive/prompt_sensitivity/<MODEL_SAFE>__<variant>.jsonl
```

Then run `scripts/prompt_sensitivity_analysis.py` locally to produce
the comparison tables and chart for report Section 19.
